In [0]:
from pyspark.sql.functions import (
    col, trim, initcap, lower, when, coalesce, expr,
    to_date, current_date, count, lit, current_timestamp,
    row_number, avg, regexp_replace
)
from pyspark.sql.window import Window

spark.sql("CREATE SCHEMA IF NOT EXISTS main.edu_quarantine")
print("✓ Schema quarantine listo")

In [0]:
from pyspark.sql.functions import (
    col)
df = spark.table("main.edu_lakehouse.bronze_alumnos")
total = df.count()

# Cuarentena 1: campos críticos nulos
nulos = df.filter(col("carrera").isNull() | col("anio_ingreso").isNull())
nulos_count = nulos.count()
(nulos
    .withColumn("motivo_rechazo", lit("CQ-01: campo crítico nulo (carrera o anio_ingreso)"))
    .withColumn("timestamp_rechazo", current_timestamp())
    .write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("main.edu_quarantine.cuarentena_alumnos")
)

# Cuarentena 2: alumnos dados de baja
bajas = df.filter(col("estado") == "baja")
bajas_count = bajas.count()
(bajas
    .withColumn("motivo_rechazo", lit("CQ-02: alumno dado de baja"))
    .withColumn("timestamp_rechazo", current_timestamp())
    .write.format("delta").mode("append")
    .saveAsTable("main.edu_quarantine.cuarentena_alumnos")
)

# Silver: solo activos, sin nulos, sin duplicados
w = Window.partitionBy("nombre_clean").orderBy("alumno_id")
silver_alumnos = (
    df
    .filter(col("carrera").isNotNull() & col("anio_ingreso").isNotNull())
    .filter(col("estado") == "activo")
    .withColumn("nombre_clean", initcap(trim(col("nombre"))))
    .withColumn("rn", row_number().over(w))
    .filter(col("rn") == 1)
    .select(
        col("alumno_id"),
        col("nombre_clean").alias("nombre"),
        trim(col("carrera")).alias("carrera"),
        col("anio_ingreso")
    )
)

unicos = silver_alumnos.count()
duplicados_count = df.filter(
    col("carrera").isNotNull() & col("anio_ingreso").isNotNull() & (col("estado") == "activo")
).count() - unicos

(silver_alumnos.write.format("delta")
    .mode("overwrite")
    .saveAsTable("main.edu_lakehouse.silver_alumnos"))

kpi_completitud = int((total - nulos_count) / total * 10000) / 100
kpi_unicidad = int(unicos / total * 10000) / 100

print(f"Total Bronze:          {total}")
print(f"Nulos rechazados:      {nulos_count}  → cuarentena (CQ-01)")
print(f"Bajas rechazadas:      {bajas_count}  → cuarentena (CQ-02)")
print(f"Duplicados:            {duplicados_count}")
print(f"Silver final:          {unicos}")
print(f"\nKPI Completitud:    {kpi_completitud}%  (óptimo > 99.5%)")
print(f"KPI Unicidad:       {kpi_unicidad}%  (óptimo = 100%)")
silver_alumnos.show(truncate=False)

In [0]:
from pyspark.sql.functions import regexp_replace

df = spark.table("main.edu_lakehouse.bronze_materias")
total = df.count()

silver_materias = (
    df
    .withColumn("nombre_norm",
        lower(regexp_replace(regexp_replace(regexp_replace(regexp_replace(
            regexp_replace(trim(col("nombre")),
            "[áàäâÁÀÄÂ]", "a"),
            "[éèëêÉÈËÊ]", "e"),
            "[íìïîÍÌÏÎ]", "i"),
            "[óòöôÓÒÖÔ]", "o"),
            " 1$", " i"))  # normaliza "Matematica 1" → "matematica i"
    )
    .withColumn("rn", row_number().over(
        Window.partitionBy("nombre_norm").orderBy("materia_id")
    ))
    .filter(col("rn") == 1)
    .select(
        col("materia_id"),
        initcap(trim(col("nombre"))).alias("nombre"),
        col("docente")
    )
)

duplicados_count = total - silver_materias.count()

(silver_materias.write.format("delta")
    .mode("overwrite")
    .saveAsTable("main.edu_lakehouse.silver_materias"))

kpi_unicidad = int(silver_materias.count() / total * 10000) / 100

print(f"Total Bronze:      {total}")
print(f"Duplicados:        {duplicados_count}  → eliminados (M04, M08, M09)")
print(f"Silver final:      {silver_materias.count()}")
print(f"\nKPI Unicidad:   {kpi_unicidad}%  (óptimo = 100%)")
silver_materias.show(truncate=False)

In [0]:
alumnos_ids = spark.table("main.edu_lakehouse.silver_alumnos").select("alumno_id")
materias_ids = spark.table("main.edu_lakehouse.silver_materias").select("materia_id")

# Verificar huérfanos de materia
huerfanos_materias = (
    spark.table("main.edu_lakehouse.silver_notas")
    .join(materias_ids, "materia_id", "left_anti")
)
print(f"Notas con materia_id inválido: {huerfanos_materias.count()}")
huerfanos_materias.show(truncate=False)

# Enviar a cuarentena
if huerfanos_materias.count() > 0:
    (huerfanos_materias
        .withColumn("motivo_rechazo", lit("CQ-01: materia_id no existe en catálogo maestro silver_materias"))
        .withColumn("timestamp_rechazo", current_timestamp())
        .write.format("delta").mode("overwrite")
        .saveAsTable("main.edu_quarantine.cuarentena_notas_materia_invalida")
    )

    # Limpiar silver_notas
    silver_notas_limpia = (
        spark.table("main.edu_lakehouse.silver_notas")
        .join(materias_ids, "materia_id", "inner")
    )
    (silver_notas_limpia.write.format("delta")
        .mode("overwrite")
        .saveAsTable("main.edu_lakehouse.silver_notas"))
    print(f"✓ silver_notas actualizada: {silver_notas_limpia.count()} registros")
else:
    print("✓ No hay notas huérfanas de materia")

In [0]:
df = spark.table("main.edu_lakehouse.bronze_notas")
total = df.count()

df_dedup = df.dropDuplicates()
duplicados_count = total - df_dedup.count()

df_typed = df_dedup.withColumn("nota_num", expr("try_cast(nota as float)"))

df_clasificado = (
    df_typed
    .withColumn(
        "estado_nota",
        when(lower(col("nota")) == "ausente", "ausente")
        .when(col("nota").isNull(), "pendiente")
        .when(col("nota_num") < 0, "error_carga")
        .when(col("nota_num") > 10, "error_carga")
        .otherwise("valida")
    )
    .withColumn("fecha_parseada",
        coalesce(
            expr("try_to_date(fecha_examen, 'yyyy-MM-dd')"),
            expr("try_to_date(fecha_examen, 'dd/MM/yyyy')")
        )
    )
    .withColumn(
        "fecha_futura",
        when(col("fecha_parseada") > current_date(), True).otherwise(False)
    )
)

rechazados = df_clasificado.filter(
    (col("estado_nota") == "error_carga") |
    (col("fecha_futura") == True) |
    (col("fecha_parseada").isNull())
)
rechazados_count = rechazados.count()

(rechazados
    .withColumn("motivo_rechazo",
        when(col("fecha_parseada").isNull(), "CF-01: fecha imposible o formato inválido")
        .when(col("fecha_futura") == True, "VL-01: fecha futura")
        .otherwise("VL-01: nota fuera de rango [0-10]"))
    .withColumn("timestamp_rechazo", current_timestamp())
    .write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("main.edu_quarantine.cuarentena_notas")
)

silver_notas = (
    df_clasificado
    .filter(col("estado_nota") != "error_carga")
    .filter(col("fecha_futura") == False)
    .filter(col("fecha_parseada").isNotNull())
    .withColumn(
        "nota_final",
        when(col("estado_nota") == "valida", col("nota_num")).otherwise(None)
    )
    .select(
        col("alumno_id"),
        col("materia_id"),
        col("fecha_parseada").alias("fecha_examen"),
        col("nota_final").alias("nota"),
        col("estado_nota")
    )
)

silver_count = silver_notas.count()
validas = df_clasificado.filter(col("estado_nota") == "valida").count()
nulos_nota = df_dedup.filter(col("nota").isNull()).count()

(silver_notas.write.format("delta")
    .mode("overwrite")
    .saveAsTable("main.edu_lakehouse.silver_notas"))

kpi_completitud = int((total - nulos_nota) / total * 10000) / 100
kpi_unicidad    = int(df_dedup.count() / total * 10000) / 100
kpi_validez     = int(validas / silver_count * 10000) / 100

print(f"Total Bronze:        {total}")
print(f"Duplicados:          {duplicados_count}  → eliminados")
print(f"Rechazados:          {rechazados_count}  → cuarentena_notas")
print(f"Silver final:        {silver_count}")
print(f"\nKPI Completitud:  {kpi_completitud}%  (óptimo > 99.5%)")
print(f"KPI Unicidad:     {kpi_unicidad}%   (óptimo = 100%)")
print(f"KPI Validez:      {kpi_validez}%  (óptimo > 97.5%)")
print("\nDistribución por estado:")
silver_notas.groupBy("estado_nota").count().show()

In [0]:
df = spark.table("main.edu_lakehouse.bronze_asistencia")
total = df.count()

df_dedup = df.dropDuplicates()
duplicados_count = total - df_dedup.count()

df_typed = (
    df_dedup
    .withColumn("fecha_clean",
        coalesce(
            expr("try_to_date(fecha, 'yyyy-MM-dd')"),
            expr("try_to_date(fecha, 'dd/MM/yyyy')")
        ))
    .withColumn("presente_clean",
        when(col("presente").isin("1", "si", "SI", "sí"), 1)
        .when(col("presente").isin("0", "no", "NO"), 0)
        .otherwise(None))
)

rechazados = df_typed.filter(
    col("fecha_clean").isNull() |
    col("presente_clean").isNull() |
    (col("fecha_clean") > current_date())
)
rechazados_count = rechazados.count()

(rechazados
    .withColumn("motivo_rechazo",
        when(col("fecha_clean").isNull(), "CF-01: formato de fecha inválido")
        .when(col("fecha_clean") > current_date(), "VL-01: fecha futura")
        .otherwise("CQ-01: campo presente nulo o inválido"))
    .withColumn("timestamp_rechazo", current_timestamp())
    .write.format("delta").mode("overwrite")
    .saveAsTable("main.edu_quarantine.cuarentena_asistencia")
)

silver_asistencia = (
    df_typed
    .filter(col("fecha_clean").isNotNull())
    .filter(col("presente_clean").isNotNull())
    .filter(col("fecha_clean") <= current_date())
    .select(
        col("alumno_id"),
        col("materia_id"),
        col("fecha_clean").alias("fecha"),
        col("presente_clean").alias("presente")
    )
)

silver_count = silver_asistencia.count()
nulos_presente = df_dedup.filter(col("presente").isNull()).count()
formatos_invalidos = df_typed.filter(col("fecha_clean").isNull()).count()

(silver_asistencia.write.format("delta")
    .mode("overwrite")
    .saveAsTable("main.edu_lakehouse.silver_asistencia"))

kpi_completitud = int((total - nulos_presente) / total * 10000) / 100
kpi_formato     = int((total - formatos_invalidos) / total * 10000) / 100
kpi_unicidad    = int(df_dedup.count() / total * 10000) / 100

print(f"Total Bronze:           {total}")
print(f"Duplicados:             {duplicados_count}  → eliminados")
print(f"Rechazados:             {rechazados_count}  → cuarentena_asistencia")
print(f"Silver final:           {silver_count}")
print(f"\nKPI Completitud:     {kpi_completitud}%  (óptimo > 99.5%)")
print(f"KPI Formato válido:  {kpi_formato}%  (óptimo > 98.0%)")
print(f"KPI Unicidad:        {kpi_unicidad}%  (óptimo = 100%)")

In [0]:
print("=== TABLAS SILVER ===")
spark.sql("SHOW TABLES IN main.edu_lakehouse")\
    .filter(col("tableName").startswith("silver")).show(truncate=False)

print("\n=== TABLAS CUARENTENA ===")
spark.sql("SHOW TABLES IN main.edu_quarantine").show(truncate=False)

print("\n=== Alumnos rechazados ===")
spark.table("main.edu_quarantine.cuarentena_alumnos").show(truncate=False)

print("\n=== Notas rechazadas ===")
spark.table("main.edu_quarantine.cuarentena_notas").show(truncate=False)

print("\n=== Asistencia rechazada ===")
spark.table("main.edu_quarantine.cuarentena_asistencia").show(truncate=False)

## MERGE Delta con validación referencial
Simulamos la llegada de un lote incremental de notas. Dos casos:
- Alumno 6: tenía `estado_nota = pendiente`, ahora llega su nota real.
- Alumno 99: viene en el archivo pero no existe en el catálogo maestro de alumnos.

La capa Silver valida integridad referencial antes del MERGE. Los registros que no pasan
la validación van a cuarentena con motivo CQ-01. Solo los datos válidos entran al MERGE.

In [0]:
spark.sql("DROP TABLE IF EXISTS main.edu_quarantine.cuarentena_integridad_referencial")
spark.sql("DROP TABLE IF EXISTS main.edu_quarantine.cuarentena_asistencia_referencial")

from pyspark.sql import Row

nuevos_datos = spark.createDataFrame([
    Row(alumno_id=6,  materia_id="M06", fecha_examen="2024-07-15", nota=7.0, estado_nota="valida"),
    Row(alumno_id=99, materia_id="M01", fecha_examen="2024-07-15", nota=8.0, estado_nota="valida"),
])

alumnos_validos = spark.table("main.edu_lakehouse.silver_alumnos").select("alumno_id")
datos_validados = nuevos_datos.join(alumnos_validos, "alumno_id", "inner")
datos_rechazados = nuevos_datos.join(alumnos_validos, "alumno_id", "left_anti")

print(f"Registros a procesar: {datos_validados.count()}")
print(f"Rechazados por integridad referencial: {datos_rechazados.count()}")
datos_rechazados.show()

(datos_rechazados
    .withColumn("motivo_rechazo", lit("CQ-01: alumno_id no existe en catálogo maestro"))
    .withColumn("timestamp_rechazo", current_timestamp())
    .write.format("delta").mode("overwrite")
    .saveAsTable("main.edu_quarantine.cuarentena_integridad_referencial")
)
print("✓ Rechazados enviados a cuarentena_integridad_referencial")

datos_validados.createOrReplaceTempView("notas_nuevas")

spark.sql("""
    MERGE INTO main.edu_lakehouse.silver_notas AS destino
    USING notas_nuevas AS origen
    ON destino.alumno_id = origen.alumno_id 
    AND destino.materia_id = origen.materia_id
    WHEN MATCHED AND destino.estado_nota = 'pendiente' THEN
        UPDATE SET 
            destino.nota = origen.nota,
            destino.estado_nota = origen.estado_nota
    WHEN NOT MATCHED THEN
        INSERT (alumno_id, materia_id, fecha_examen, nota, estado_nota)
        VALUES (origen.alumno_id, origen.materia_id, origen.fecha_examen, origen.nota, origen.estado_nota)
""")

print("\n✓ MERGE ejecutado")
print("\nEstado de notas después del MERGE:")
spark.table("main.edu_lakehouse.silver_notas").orderBy("alumno_id").show(truncate=False)

In [0]:
print("=== Historial de versiones de silver_notas ===")
spark.sql("DESCRIBE HISTORY main.edu_lakehouse.silver_notas")\
    .select("version", "timestamp", "operation")\
    .show(truncate=False)

# Usamos la versión anterior al último MERGE en vez de la versión 0
print("\n=== Versión antes del MERGE ===")
spark.sql("DESCRIBE HISTORY main.edu_lakehouse.silver_notas")\
    .select("version", "operation")\
    .filter(col("operation") == "MERGE")\
    .show(truncate=False)

# Tomamos la versión del MERGE y consultamos la anterior
merge_version = spark.sql("DESCRIBE HISTORY main.edu_lakehouse.silver_notas")\
    .filter(col("operation") == "MERGE")\
    .select("version")\
    .first()[0]

version_antes = merge_version - 1

print(f"\n=== Versión {version_antes} — estado antes del MERGE ===")
spark.sql(f"SELECT * FROM main.edu_lakehouse.silver_notas VERSION AS OF {version_antes}")\
    .filter(col("alumno_id") == 6)\
    .show(truncate=False)

print("\n=== Versión actual — después del MERGE ===")
spark.sql("SELECT * FROM main.edu_lakehouse.silver_notas")\
    .filter(col("alumno_id") == 6)\
    .show(truncate=False)

print("\n→ Delta conserva el historial de versiones para auditoría")

In [0]:
alumnos_ids = spark.table("main.edu_lakehouse.silver_alumnos").select("alumno_id")
materias_ids = spark.table("main.edu_lakehouse.silver_materias").select("materia_id")

huerfanos_notas = spark.table("main.edu_lakehouse.silver_notas")\
    .join(alumnos_ids, "alumno_id", "left_anti")
huerfanos_asistencia = spark.table("main.edu_lakehouse.silver_asistencia")\
    .join(alumnos_ids, "alumno_id", "left_anti")
huerfanos_materias = spark.table("main.edu_lakehouse.silver_notas")\
    .join(materias_ids, "materia_id", "left_anti")

print(f"Notas huérfanas (alumno):   {huerfanos_notas.count()}")
print(f"Asistencia huérfana:        {huerfanos_asistencia.count()}")
print(f"Notas huérfanas (materia):  {huerfanos_materias.count()}")

huerfanos_notas.show(truncate=False)
huerfanos_asistencia.show(truncate=False)
huerfanos_materias.show(truncate=False)

spark.sql("DROP TABLE IF EXISTS main.edu_quarantine.cuarentena_integridad_referencial")
spark.sql("DROP TABLE IF EXISTS main.edu_quarantine.cuarentena_asistencia_referencial")

(huerfanos_notas
    .withColumn("motivo_rechazo", lit("CQ-01: alumno_id no existe en catálogo maestro silver_alumnos"))
    .withColumn("timestamp_rechazo", current_timestamp())
    .write.format("delta").mode("overwrite")
    .saveAsTable("main.edu_quarantine.cuarentena_integridad_referencial")
)
print(f"✓ {huerfanos_notas.count()} notas huérfanas → cuarentena")

(huerfanos_asistencia
    .withColumn("motivo_rechazo", lit("CQ-01: alumno_id no existe en catálogo maestro silver_alumnos"))
    .withColumn("timestamp_rechazo", current_timestamp())
    .write.format("delta").mode("overwrite")
    .saveAsTable("main.edu_quarantine.cuarentena_asistencia_referencial")
)
print(f"✓ {huerfanos_asistencia.count()} registros asistencia huérfanos → cuarentena")

silver_notas_limpia = spark.table("main.edu_lakehouse.silver_notas")\
    .join(alumnos_ids, "alumno_id", "inner")
(silver_notas_limpia.write.format("delta")
    .mode("overwrite")
    .saveAsTable("main.edu_lakehouse.silver_notas"))
print(f"✓ silver_notas actualizada: {silver_notas_limpia.count()} registros")

silver_asistencia_limpia = spark.table("main.edu_lakehouse.silver_asistencia")\
    .join(alumnos_ids, "alumno_id", "inner")
(silver_asistencia_limpia.write.format("delta")
    .mode("overwrite")
    .saveAsTable("main.edu_lakehouse.silver_asistencia"))
print(f"✓ silver_asistencia actualizada: {silver_asistencia_limpia.count()} registros")

print("\n=== Verificación final ===")
restantes_notas = spark.table("main.edu_lakehouse.silver_notas")\
    .join(alumnos_ids, "alumno_id", "left_anti").count()
restantes_asistencia = spark.table("main.edu_lakehouse.silver_asistencia")\
    .join(alumnos_ids, "alumno_id", "left_anti").count()
print(f"Notas huérfanas restantes:      {restantes_notas}")
print(f"Asistencia huérfana restante:   {restantes_asistencia}")
print("✓ Integridad referencial OK")

In [0]:
from pyspark.sql import Row

kpis = [
    Row(tabla="alumnos",    dimension="Completitud", valor=88.88, umbral_optimo=99.5,  estado="CRITICO"),
    Row(tabla="alumnos",    dimension="Unicidad",    valor=66.66, umbral_optimo=100.0, estado="CRITICO"),
    Row(tabla="materias",   dimension="Unicidad",    valor=66.66, umbral_optimo=100.0, estado="CRITICO"),
    Row(tabla="notas",      dimension="Completitud", valor=93.54, umbral_optimo=99.5,  estado="CRITICO"),
    Row(tabla="notas",      dimension="Unicidad",    valor=96.77, umbral_optimo=100.0, estado="CRITICO"),
    Row(tabla="notas",      dimension="Validez",     valor=92.59, umbral_optimo=97.5,  estado="CRITICO"),
    Row(tabla="asistencia", dimension="Completitud", valor=98.33, umbral_optimo=99.5,  estado="ALERTA"),
    Row(tabla="asistencia", dimension="Formato",     valor=100.0, umbral_optimo=98.0,  estado="OPTIMO"),
    Row(tabla="asistencia", dimension="Unicidad",    valor=96.66, umbral_optimo=100.0, estado="CRITICO"),
]

df_kpis = spark.createDataFrame(kpis)
df_kpis.show(truncate=False)

(df_kpis.write.format("delta")
    .mode("overwrite")
    .saveAsTable("main.edu_lakehouse.gold_kpis_calidad"))
print("✓ gold_kpis_calidad actualizada con valores reales")